In [ ]:
import json
import os
import re
import argparse
import traceback
from pathlib import Path

# ── MedRAG ──────────────────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/vast/palmer/apps/avx2/software/Java/11.0.16"

from src.medrag import MedRAG


# ── Prompt templates ─────────────────────────────────────────────────────────

FORMAT_PUBMEDQA = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<yes|no|maybe>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

FORMAT_MEDQA = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<single letter>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

FORMAT_OPEN = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<concise direct answer>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

DATASET_FORMAT = {
    "PubMedQA": FORMAT_PUBMEDQA,
    "MedQA-US": FORMAT_MEDQA,
    "BioASQ":   FORMAT_OPEN,
    "MediQA":   FORMAT_OPEN,
}


def get_prompt_suffix(dataset: str) -> str:
    return DATASET_FORMAT.get(dataset, FORMAT_OPEN)


# ── Output parser ─────────────────────────────────────────────────────────────

def parse_llm_output(raw: str) -> tuple:
    """
    Returns (llm_answer, llm_explanation, parse_error).
    Tries several strategies to recover valid JSON from the raw string.
    """
    # Strategy 1: strip markdown fences
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r"```\s*$", "", cleaned.strip(), flags=re.MULTILINE)
    cleaned = cleaned.strip()

    # Strategy 2: find first {...} block
    if not cleaned.startswith("{"):
        m = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if m:
            cleaned = m.group(0)

    try:
        obj = json.loads(cleaned)
        answer = str(obj.get("answer", "")).strip()
        explanation = obj.get("explanation", [])
        if not isinstance(explanation, list):
            explanation = []
        return answer, explanation, False
    except json.JSONDecodeError:
        return raw.strip(), [], True


# ── Main pipeline ─────────────────────────────────────────────────────────────

def run_pipeline(
    input_path,
    output_path,
    start_idx=0,
    end_idx=None,
    k=32,
):
    # Initialise MedRAG once (expensive)
    print("Initialising MedRAG ...")
    medrag = MedRAG(
        llm_name="OpenAI/gpt-4o",
        rag=True,
        retriever_name="RRF-4",
        corpus_name="MedCorp",
        corpus_cache=True,
        adaptive_k=True, 
        use_llm_k=True
    )
    print("MedRAG ready.\n")

    # Load dataset
    records = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    subset = records[start_idx:end_idx]
    total = len(subset)
    print(f"Processing {total} records (indices {start_idx}-{start_idx + total - 1}) ...\n")

    done_questions = set()
    if Path(output_path).exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        done_questions.add(json.loads(line)["question"])
                    except Exception:
                        pass
        print(f"Resuming - {len(done_questions)} already done.\n")

    out_f = open(output_path, "a", encoding="utf-8")

    for i, rec in enumerate(subset):
        global_idx = start_idx + i
        dataset      = rec.get("dataset", "Unknown")
        question     = rec.get("question", "")
        ground_truth = rec.get("answer", "")

        if question in done_questions:
            print(f"[{global_idx+1}/{start_idx+total}] SKIP (already done): {question[:60]}")
            continue

        print(f"[{global_idx+1}/{start_idx+total}] [{dataset}] {question[:80]}")

        try:
            raw_output, snippets, scores = medrag.answer(
                question=question,
                format_instruction=get_prompt_suffix(dataset), 
                seed=42,
                max_tokens=2048, 
                top_p=1,
            )
        except Exception:
            print(f"  ERROR calling MedRAG:\n{traceback.format_exc()}")
            raw_output, snippets, scores = "", [], []

        llm_answer, llm_explanation, parse_error = parse_llm_output(raw_output)

        result = {
            "dataset":         dataset,
            "question":        question,
            "ground_truth":    ground_truth,
            "llm_answer":      llm_answer,
            "llm_explanation": llm_explanation,
            "raw_output":      raw_output,
            "snippets":        snippets,
            "metrics":         {},   # filled later by evaluation pipeline
        }

        out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
        out_f.flush()

        status = "PARSE_ERROR" if parse_error else "OK"
        print(f"  -> {status}  answer={repr(llm_answer[:60])}")

    out_f.close()
    print(f"\nDone. Results saved to {output_path}")


# ── CLI ───────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(description="Run MedRAG over medical_qa_1000.jsonl")
    parser.add_argument("--input",  default="medical_qa_1000_clean.jsonl")
    parser.add_argument("--output", default="medrag_results_adaptK.jsonl")
    parser.add_argument("--start",  type=int, default=0)
    parser.add_argument("--end",    type=int, default=None)
    args, unknown = parser.parse_known_args()

    run_pipeline(
        input_path  = args.input,
        output_path = args.output,
        start_idx   = args.start,
        end_idx     = args.end
    )

if __name__ == "__main__":
    main()

Initialising MedRAG ...


No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.


No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
Initializing the document extracter...
Initialization finished!
MedRAG ready.

Processing 300 records (indices 0-299) ...

Resuming - 180 already done.

[1/300] SKIP (already done): what are autoimmune blood disorders
[2/300] SKIP (already done): The

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
"""
MedRAG Pipeline
Runs MedRAG over medical_qa_1000.jsonl and saves results to a new JSONL file.
"""

import json
import os
import re
import argparse
import traceback
from pathlib import Path

# ── MedRAG ──────────────────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/vast/palmer/apps/avx2/software/Java/11.0.16"

from src.medrag import MedRAG


# ── Prompt templates ─────────────────────────────────────────────────────────

FORMAT_PUBMEDQA = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<yes|no|maybe>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

FORMAT_MEDQA = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<single letter>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

FORMAT_OPEN = """\
Respond ONLY with a JSON object in this exact format.
No preamble, no markdown:
{
  "answer": "<concise direct answer>",
  "explanation": [
    {"sentence": "<sentence text>", "citations": [<list of snippet indices>]},
    ...
  ]
}"""

DATASET_FORMAT = {
    "PubMedQA": FORMAT_PUBMEDQA,
    "MedQA-US": FORMAT_MEDQA,
    "BioASQ":   FORMAT_OPEN,
    "MediQA":   FORMAT_OPEN,
}


def get_prompt_suffix(dataset: str) -> str:
    return DATASET_FORMAT.get(dataset, FORMAT_OPEN)


# ── Output parser ─────────────────────────────────────────────────────────────

def parse_llm_output(raw: str) -> tuple:
    """
    Returns (llm_answer, llm_explanation, parse_error).
    Tries several strategies to recover valid JSON from the raw string.
    """
    # Strategy 1: strip markdown fences
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r"```\s*$", "", cleaned.strip(), flags=re.MULTILINE)
    cleaned = cleaned.strip()

    # Strategy 2: find first {...} block
    if not cleaned.startswith("{"):
        m = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if m:
            cleaned = m.group(0)

    try:
        obj = json.loads(cleaned)
        answer = str(obj.get("answer", "")).strip()
        explanation = obj.get("explanation", [])
        if not isinstance(explanation, list):
            explanation = []
        return answer, explanation, False
    except json.JSONDecodeError:
        return raw.strip(), [], True


# ── Main pipeline ─────────────────────────────────────────────────────────────

def run_pipeline(
    input_path,
    output_path,
    start_idx=0,
    end_idx=None,
    k=32,
):
    # Initialise MedRAG once (expensive)
    print("Initialising MedRAG ...")
    medrag = MedRAG(
        llm_name="OpenAI/gpt-4o",
        rag=True,
        retriever_name="RRF-4",
        corpus_name="MedCorp",
        corpus_cache=True,
        k=32,
    )
    print("MedRAG ready.\n")

    # Load dataset
    records = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    subset = records[start_idx:end_idx]
    total = len(subset)
    print(f"Processing {total} records (indices {start_idx}-{start_idx + total - 1}) ...\n")

    # Resume: collect already-processed questions to skip
    done_questions = set()
    if Path(output_path).exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        done_questions.add(json.loads(line)["question"])
                    except Exception:
                        pass
        print(f"Resuming - {len(done_questions)} already done.\n")

    out_f = open(output_path, "a", encoding="utf-8")

    for i, rec in enumerate(subset):
        global_idx = start_idx + i
        dataset      = rec.get("dataset", "Unknown")
        question     = rec.get("question", "")
        ground_truth = rec.get("answer", "")

        if question in done_questions:
            print(f"[{global_idx+1}/{start_idx+total}] SKIP (already done): {question[:60]}")
            continue

        print(f"[{global_idx+1}/{start_idx+total}] [{dataset}] {question[:80]}")

        try:
            raw_output, snippets, scores = medrag.answer(
                question=question,
                format_instruction=get_prompt_suffix(dataset),  
                seed=42,
                max_tokens=2048, 
                top_p=1,
                k=32,
            )
        except Exception:
            print(f"  ERROR calling MedRAG:\n{traceback.format_exc()}")
            raw_output, snippets, scores = "", [], []

        llm_answer, llm_explanation, parse_error = parse_llm_output(raw_output)

        result = {
            "dataset":         dataset,
            "question":        question,
            "ground_truth":    ground_truth,
            "llm_answer":      llm_answer,
            "llm_explanation": llm_explanation,
            "raw_output":      raw_output,
            "snippets":        snippets,
            "metrics":         {},   # filled later by evaluation pipeline
        }

        out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
        out_f.flush()

        status = "PARSE_ERROR" if parse_error else "OK"
        print(f"  -> {status}  answer={repr(llm_answer[:60])}")

    out_f.close()
    print(f"\nDone. Results saved to {output_path}")


# ── CLI ───────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(description="Run MedRAG over medical_qa_1000_clean.jsonl")
    parser.add_argument("--input",  default="medical_qa_1000_clean.jsonl")
    parser.add_argument("--output", default="medrag_results_fixed.jsonl")
    parser.add_argument("--start",  type=int, default=0)
    parser.add_argument("--end",    type=int, default=None)
    args, unknown = parser.parse_known_args()

    run_pipeline(
        input_path  = args.input,
        output_path = args.output,
        start_idx   = args.start,
        end_idx     = args.end
    )

if __name__ == "__main__":
    main()

Initialising MedRAG ...


No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/facebook_contriever. Creating a new one with MEAN pooling.


No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/allenai_specter. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/sj868/.cache/torch/sentence_transformer